# 01 — EDA: Product Images

**Question:** what does our labelled electronics image dataset look like, and are there class-imbalance or quality issues that will bias the classifier?

**Inputs:** `data/raw/products/` (downloaded via `data/download.py`)

**Outputs:** class-balance chart, size distribution, sample grid saved to `docs/figures/`.


## Setup

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
RAW = Path('../data/raw/products')
FIG = Path('../docs/figures'); FIG.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_rows', 20)

## 1. Load the manifest

We expect `products/*/*.jpg` where the parent folder is the label.

In [ ]:
rows = []
for cls_dir in sorted(RAW.glob('*')):
    if not cls_dir.is_dir(): continue
    for img in cls_dir.glob('*.jpg'):
        rows.append({'label': cls_dir.name, 'path': str(img)})
df = pd.DataFrame(rows)
print(f'{len(df):,} images across {df.label.nunique()} classes')
df.head()

## 2. Class balance

Key question: is any class < 5% of the dataset (would need oversampling)?

In [ ]:
counts = df.label.value_counts()
ax = counts.plot(kind='barh', figsize=(6, max(3, 0.3*len(counts))), color='#7c5cff')
ax.set_title('Images per class'); ax.set_xlabel('count'); plt.tight_layout()
plt.savefig(FIG/'01_class_balance.png', dpi=140)
counts

## 3. Image size distribution

We'll resize to 224×224 for ResNet50 — but very small sources compress poorly.

In [ ]:
import random
random.seed(SEED)
sample = df.sample(min(500, len(df)), random_state=SEED)
sizes = [Image.open(p).size for p in sample.path]
w, h = zip(*sizes)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].hist(w, bins=30, color='#7c5cff'); ax[0].set_title('width (px)')
ax[1].hist(h, bins=30, color='#2ee6a6'); ax[1].set_title('height (px)')
plt.tight_layout(); plt.savefig(FIG/'01_size_distribution.png', dpi=140)
pd.Series(w).describe().round(0)

## 4. Sample grid

Sanity check — do the labels actually match the images?

In [ ]:
import math
cols = 6
classes = df.label.unique()[:cols]
rows_ = 3
fig, axes = plt.subplots(rows_, cols, figsize=(cols*2, rows_*2))
for c, cls in enumerate(classes):
    imgs = df[df.label==cls].sample(rows_, random_state=SEED).path.tolist()
    for r, p in enumerate(imgs):
        axes[r,c].imshow(Image.open(p)); axes[r,c].axis('off')
        if r==0: axes[r,c].set_title(cls, fontsize=9)
plt.tight_layout(); plt.savefig(FIG/'01_sample_grid.png', dpi=140)

## Findings

- **Class balance:** _(fill in after run)_
- **Sizes:** median ~ ? × ?, tail below 128px will be filtered before training.
- **Label quality:** grid inspection — _(fill in)_.

**Implications for modelling:**
1. Use class-weighted CE loss if imbalance > 3:1.
2. Resize with letterbox padding to keep aspect ratio.
3. Manual re-label any obviously wrong sample flagged in the grid.